In [5]:
# drawing utils
%matplotlib notebook
import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib import ticker, colors
from matplotlib import animation
# from matplotlib.colors import hsv_to_rgb
#matplotlib.rcParams['text.usetex'] = True

plt.rcParams['xtick.direction']='in'
plt.rcParams['ytick.direction']='in'

from cycler import cycler

import os
import re


# math utils
import numpy as np

# statistics utils
import pandas as pd

def get_key_list(folder_path):
    key_list = []
    for i in range(len(filepath)):
        for j in range(len(filepath[i])):
            if filepath[i][j] == '_':
                start_index = j
            elif filepath[i][j] == 'p':
                end_index = j
            key_list.append(filepath[i][start_index+1:end_index+2])
            return key_list

def FileNameParse(folder_path， key_list):
    file_path_list = []
    for maindir, subdie, file_name_list in os.walk(folder_path):
        for file_name in file_name_list:
            filepath.append(os.path.join(maindir, file_name))  
    DelayList = []
    start_index = 0
    end_index = 0
    FileDict = {}
    for file_path in file_path_list:
        for key in key_list:
            if key in file_path:
                FileDict.setdefault(key,[]).append(file_path)
    return FileDict

#数据平均、截取
def parse_data(file_path, xmin, xmax,ymin, ymax,norm = False):
    rdata = np.loadtxt(file_path)
    ylist = rdata[0,1:]
    xlist = rdata[1:,0]
    xmin_index = np.searchsorted(xlist, xmin)
    ymax_index = np.searchsorted(xlist, xmax)
    xlist = xlist[xmin_index:xmax_index]
    ymin_index = np.searchsorted(ylist, ymin)
    ymax_index = np.searchsorted(ylist, ymax)
    ylist = ylist[ymin_index:ymax_index]
    if Norm == True:
        data_max=np.amax(np.absolute(data))
        data = np.transpose(data/data_max)*-1
    data_max = data.max()
    data_min = data.min()
    return ylist, xlist, data, data_max, data_min

def parse_data_list(file_list, xmin, xmax,ymin, ymax, norm = False):
    rdata = np.loadtxt(file_list[1])
    ylist = rdata[0,1:]
    xlist = rdata[1:,0]
    xmin_index = np.searchsorted(xlist, xmin)
    ymax_index = np.searchsorted(xlist, xmax)
    xlist = xlist[xmin_index:xmax_index]
    ymin_index = np.searchsorted(ylist, ymin)
    ymax_index = np.searchsorted(ylist, ymax)
    ylist = ylist[ymin_index:ymax_index]
    data = np.zeros([xmax_index - ymin_index,xmax_index - ymin_index])
    for file_path in file_list:
        rdata = np.loadtxt(file_path)
        data = rdata [pump_min_index:pump_max_index,probe_min_index:probe_max_index]+data
    data = data/len(file_path_list)
    if Norm == True:
        data_max=np.amax(np.absolute(data))
        data = np.transpose(data/data_max)*-1
    data_max = data.max()
    data_min = data.min()
    return ylist, xlist, data, data_max, data_min

#colormap调整
def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=100):
    new_cmap = colors.LinearSegmentedColormap.from_list(
        "trunc({n},{a:.2f},{b:.2f})".format(n=cmap.name, a=minval, b=maxval),
        cmap(np.linspace(minval, maxval, n)),
    )
    return new_cmap

class twod_plot:
    sample_name = 'sample_name'
    fig_save_path=r'./'
    fig_format = 'png'
    fig_dpi = 200
    
    key_name = ''
    key_list=['key_name']
    
    colormap = 'jet'
    color_level = np.arrange(-1, 1, 0.01)
    contour_line_level = [-1,-0.95,-0.9,-0.85,-0.8,-0.7,-0.6,-0.5,-0.4,-0.3,-0.2,-0.1,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
    line_color = 'black'
    line_style = 'solid'
    
    
    plt.rcParams['xtick.direction']='in'
    plt.rcParams['ytick.direction']='in'
    xlabel = 'xlable'
    ylabel = 'ylable'
    colorbar_lable = 'colorbar_lable'
    
    xmin = 0
    xmax = 1 
    ymin = 0 
    ymax = 1
    file_list_average = False
    normalization = True
    
    file_path = ''
    folder_path = ''
    
    font = 'Arial'
    title_fontsize = 30
    lable_fontsize = 28
    tick_fontsize = 20
    colorbar_tick_fontsize 16
    fontweight = 'bold'
    
    
    def plot_2d(self, key_name = self.key_name):
        if not os.path.exists(self.filepath):
            print('File not found, program stopped')
            os._exit(0)
        ylist, xlist, data, dmax, dmin = parse_data(self.filepath, self.xmin, self.xmax, self.ymin, self.ymax, norm = normalization)
        fig,ax=plt.subplots(figsize=(10,8))
        figtitle = self.sample_name +' '+ key_name
        plt.title(figtitle,fontproperties=self.font, fontsize=self.title_fontsize, fontweight=self.fontweight)
        plt.xlabel(self.xlabel,fontproperties=self.font, fontsize=self.lable_fontsize, fontweight=self.fontweight)
        plt.ylabel(self.ylabel,fontproperties=self.font, fontsize=self.lable_fontsize, fontweight=self.fontweight)
        plt.xticks(fontproperties=self.font, fontsize=self.tick_fontsize, fontweight=self.fontweight)
        plt.yticks(fontproperties=self.font, fontsize=self.tick_fontsize, fontweight=self.fontweight)
        cmap = plt.get_cmap(self.colormap)
        trunc_cmap = truncate_colormap(cmap, 0.1, 0.9,400)
        cbar = plt.colorbar(im,)
        cbar.set_label(self.colorbar_lable, font = self.font, fontsize = self.lable_fontsize, weight=self.fontweight)
        cbar.ax.tick_params(labelsize=self.colorbar_tick_fontsize)
        im = ax.contourf(xlist, ylist, data, levels=self.color_level, cmap=trunc_cmap, extend="neither")
        plt.contour(pump_wavelength, probe_wavlength, data, levels=self.contour_line_level, 
                    colors=self.line_color, linestyles=self.line_style)
        if not os.path.exists('./'+figfolder_path):
            os.makedirs('./'+figfolder_path)  
        figpath=r'./'+figfolder_path+r'/'+sample_name+'_'+delay+'.'+fig_format
        fig.savefig(figpath, dpi=fig_dpi, format=self.fig_format)
        return 0
    
    def plot_list_2d(self, key_list = self.key_list):
        file_dict = FileNameParse(self.folder_path, key_list)
        for key_name in key_list:
            plot_2d(key_name = key_name)
    

def plot_pp_2d(filepath_list, sample_name, figfolder_path, delay, pump_min, pump_max,probe_min, probe_max):
    fig,ax=plt.subplots(figsize=(10,8))
    figtitle = sample_name +'   '+ delay 
    probe_wavlength, pump_wavelength,delay_list_2, data, dmax, dmin = parse_pp_data(filepath_list, pump_min, pump_max,probe_min, probe_max)
    plt.title(figtitle,fontproperties='Arial', fontsize = 30, fontweight='bold')
    plt.xlabel('Wavelength (nm )',fontproperties='Arial', fontsize=28, fontweight='bold')
    plt.ylabel('Wavelength (nm )',fontproperties='Arial', fontsize=28, fontweight='bold')
    plt.xticks(fontproperties='Arial', size=20, weight='bold')
    plt.yticks(fontproperties='Arial', size=20, weight='bold')
    levels = np.arange(-1, 1, 0.01)
    level2 = [-1,-0.95,-0.9,-0.85,-0.8,-0.7,-0.6,-0.5,-0.4,-0.3,-0.2,-0.1,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
    level3 = np.arange(-1, 1, 0.1)
    cmap = plt.get_cmap('jet')
    trunc_cmap = truncate_colormap(cmap, 0.1, 0.9,400)
    im = ax.contourf(pump_wavelength, probe_wavlength, data, levels=levels, cmap=trunc_cmap, extend="neither")
    plt.contour(pump_wavelength, probe_wavlength, data, levels=level3, colors='black', linestyles='solid')
    cbar = plt.colorbar(im,)
    cbar.set_label('-Norm. '+chr(916)+'O.D.', font = 'Arial', fontsize = 28, weight='bold')
    cbar.ax.tick_params(labelsize=16)
#    cbar = fig.colorbar(im, ax=ax, orientation='vertical', label='Normalized Delta O.D.')
#    plt.show()
    if not os.path.exists('./'+figfolder_path):
        os.makedirs('./'+figfolder_path)
    print(delay)
#    figpath = delay+'.png'
#    print(figpath)   
    figpath=r'./'+figfolder_path+r'/'+sample_name+'_'+delay+'.'+fig_format
    fig.savefig(figpath, dpi=self.fig_dpi, format=self.fig_format)
    return 0 

    
folder_path=r'C:\Users\30960\OneDrive\课题组\NIR\opv\2D\PM6\20220107-622pump-10uw'
file_dict=FileNameParse(folder_path)
#print(file_dict)   
delay_list=list(file_dict.keys())
sample_name = re.split(r'\\',folder_path)[-2]
folder_name =sample_name+'-'+re.findall(r'\\([0-9a-z-]+?)w',folder_path)[0]+'w'
pump_min = 605
pump_max = 648
probe_min = 880
probe_max = 1550
for delay in delay_list:
    print(delay)
    plot_pp_2d(file_dict[delay],sample_name,folder_name,delay, pump_min, pump_max,probe_min, probe_max)

print('end')

#fig.savefig('test.png')



4
5
